In [1]:
# Импорт библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Настройки визуализации
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Расчет критической массы и ценности социальной сети

## Постановка задачи

Компания «ГородСоцСеть» запускает социальную сеть для жителей города с населением **1 млн человек**. 

Необходимо:
1. Определить **критическую массу** — минимальное число пользователей, при котором сеть становится самоподдерживающейся.
2. Рассчитать, **через сколько месяцев** сеть достигнет критической массы.
3. Оценить **ценность сети** по закону Мэткалфа и рассчитать прибыль по месяцам.
4. Рассчитать **NPV** проекта за 36 месяцев и провести анализ чувствительности.
5. Сформулировать **стратегические рекомендации**.

## Ключевые понятия

- **Закон Мэткалфа:** ценность сети растёт как квадрат числа пользователей: V(N) = k × N².
- **Критическая масса:** число пользователей, при котором доход покрывает все затраты.
- **Сетевой эффект:** после достижения критической массы рост ускоряется с 3% до 15% в месяц.

## Исходные данные

| Параметр | Значение |
|---|---|
| Общий потенциал рынка (M) | 1 000 000 чел. |
| Начальное число пользователей (N₀) | 5 000 чел. |
| Темп прироста до критической массы | 3% в месяц |
| Темп прироста после критической массы | 15% в месяц |
| Коэффициент ценности сети (k) | 0.005 руб./пользователь² |
| Затраты на поддержание сети | 2 000 000 руб./мес. |
| Стоимость привлечения пользователя (CAC) | 500 руб. |
| Средний доход с пользователя (ARPU) | 15 руб./мес. |
| Горизонт | 36 месяцев |
| Ставка дисконтирования | 1% в месяц (12% годовых) |

In [2]:
# Ввод исходных данных
M = 1_000_000           # общий потенциал рынка, чел.
N0 = 5_000              # начальное число пользователей
growth_before = 0.03    # темп прироста до критической массы
growth_after = 0.15     # темп прироста после критической массы
k = 0.005               # коэффициент ценности сети
fixed_costs = 2_000_000 # затраты на поддержание в месяц, руб.
CAC = 500               # стоимость привлечения пользователя, руб.
ARPU = 15               # средний доход с пользователя в месяц, руб.
horizon = 36            # горизонт, месяцев
discount_rate = 0.01    # ставка дисконтирования в месяц

# Проверка
print("Параметры загружены:")
print(f"  Потенциал рынка: {M:,} чел.")
print(f"  Стартовое число пользователей: {N0:,} чел.")
print(f"  Темп роста до крит. массы: {growth_before:.0%}")
print(f"  Темп роста после крит. массы: {growth_after:.0%}")

Параметры загружены:
  Потенциал рынка: 1,000,000 чел.
  Стартовое число пользователей: 5,000 чел.
  Темп роста до крит. массы: 3%
  Темп роста после крит. массы: 15%


## Часть 1. Расчёт критической массы

### Что такое критическая масса

**Критическая масса** — минимальное число пользователей, при котором сеть становится **самоподдерживающейся**: доход от пользователей покрывает все затраты (поддержание + привлечение новых).

### Условие самоподдержания


ARPU × N ≥ Затраты на поддержание + CAC × ΔN
где:
- `ARPU × N` — месячная выручка от N пользователей;
- `ΔN` — новые пользователи за месяц (прирост);
- `CAC × ΔN` — расходы на привлечение новых пользователей.

### Как найти критическую массу

1. Задать функцию **чистой прибыли**: `Profit(N) = ARPU × N - fixed_costs - CAC × ΔN`.
2. Найти **N\***, при котором `Profit(N*) = 0` — точка перехода от убытков к прибыли.
3. Учитывая, что `ΔN = N × growth`, условие сводится к:

In [3]:
# Часть 1. Расчет критической массы

# Шаг 1: Проверим строгое условие "ARPU × N ≥ fixed_costs + CAC × ΔN"
# При ΔN = N × growth_before получаем:
# ARPU × N ≥ fixed_costs + CAC × growth × N
# N × (ARPU - CAC × growth_before) ≥ fixed_costs

denominator = ARPU - CAC * growth_before
print("Проверка строгого условия:")
print(f"  ARPU = {ARPU} руб., CAC × growth_before = {CAC * growth_before:.2f} руб.")
print(f"  Знаменатель (ARPU - CAC × growth_before) = {denominator:.2f} руб.")
print()

if denominator > 0:
    critical_mass = fixed_costs / denominator
    print(f"  Критическая масса (строгое условие): {critical_mass:,.0f} чел.")
else:
    print("  ⚠️ Знаменатель ≤ 0: критическая масса по строгому условию НЕДОСТИЖИМА.")
    print("  Причина: затраты на привлечение новых пользователей (CAC × growth)")
    print("  равны или превышают доход с этих пользователей (ARPU).")
    print("  → Компания тратит на маркетинг столько же, сколько получает.")
    print()
    print("  Используем практический подход: критическая масса = покрытие")
    print("  ФИКСИРОВАННЫХ затрат (без CAC — CAC считается инвестицией в рост).")
    print()
    critical_mass = fixed_costs / ARPU

print(f"Критическая масса (покрытие фиксированных затрат):")
print(f"  = fixed_costs / ARPU = {fixed_costs:,} / {ARPU}")
print(f"  = {critical_mass:,.0f} пользователей")
print(f"  Это {critical_mass / M * 100:.1f}% от потенциала рынка ({M:,} чел.)")

Проверка строгого условия:
  ARPU = 15 руб., CAC × growth_before = 15.00 руб.
  Знаменатель (ARPU - CAC × growth_before) = 0.00 руб.

  ⚠️ Знаменатель ≤ 0: критическая масса по строгому условию НЕДОСТИЖИМА.
  Причина: затраты на привлечение новых пользователей (CAC × growth)
  равны или превышают доход с этих пользователей (ARPU).
  → Компания тратит на маркетинг столько же, сколько получает.

  Используем практический подход: критическая масса = покрытие
  ФИКСИРОВАННЫХ затрат (без CAC — CAC считается инвестицией в рост).

Критическая масса (покрытие фиксированных затрат):
  = fixed_costs / ARPU = 2,000,000 / 15
  = 133,333 пользователей
  Это 13.3% от потенциала рынка (1,000,000 чел.)


### Результат расчёта критической массы

**Критическая масса = 133 333 пользователя** (13.3% от потенциала рынка).

### Важный вывод

При заданных параметрах **строгое условие окупаемости НЕВЫПОЛНИМО**:
- Доход с одного пользователя (ARPU) = 15 руб./мес.
- Затраты на привлечение одного нового пользователя (CAC) = 500 руб.
- При темпе роста 3% это значит, что за месяц привлекается ≈ 0.03 пользователя на каждого существующего.
- CAC × 0.03 = 500 × 0.03 = 15 руб. — то есть затраты на привлечение равны доходу.

**Практический подход:** мы считаем CAC **инвестицией в рост**, а не операционным расходом. Поэтому критическая масса определяется как точка покрытия **фиксированных затрат**:

N* = fixed_costs / ARPU = 2 000 000 / 15 = 133 333 чел.

### Симуляция роста числа пользователей

Смоделируем рост числа пользователей за 36 месяцев с учётом **смены темпа роста** при достижении критической массы:
- **До 133 333 пользователей:** рост 3% в месяц.
- **После:** рост 15% в месяц (сетевой эффект).

In [4]:
# Симуляция роста числа пользователей за 36 месяцев
users = [N0]  # список чисел пользователей по месяцам
month_critical = None  # номер месяца, когда достигнута критическая масса

for month in range(1, horizon + 1):
    prev_users = users[-1]
    
    # Определяем темп роста: медленный, пока не достигли критической массы
    if prev_users < critical_mass:
        growth = growth_before
    else:
        growth = growth_after
        if month_critical is None:
            month_critical = month
    
    # Экспоненциальный рост
    new_users = prev_users * (1 + growth)
    
    # Ограничение по потенциалу рынка
    new_users = min(new_users, M)
    
    users.append(new_users)

# Вывод результатов
print(f"Критическая масса: {critical_mass:,.0f} чел.")
if month_critical:
    print(f"Критическая масса достигнута в месяце: {month_critical}")
else:
    print(f"⚠️ Критическая масса НЕ достигнута за {horizon} месяцев")

print()
print("Динамика роста пользователей (по кварталам):")
for m in range(0, horizon + 1, 3):
    mark = "  ← КРИТ. МАССА" if month_critical and m == month_critical else ""
    print(f"  Месяц {m:2d}: {users[m]:>12,.0f} чел.{mark}")

Критическая масса: 133,333 чел.
⚠️ Критическая масса НЕ достигнута за 36 месяцев

Динамика роста пользователей (по кварталам):
  Месяц  0:        5,000 чел.
  Месяц  3:        5,464 чел.
  Месяц  6:        5,970 чел.
  Месяц  9:        6,524 чел.
  Месяц 12:        7,129 чел.
  Месяц 15:        7,790 чел.
  Месяц 18:        8,512 чел.
  Месяц 21:        9,301 чел.
  Месяц 24:       10,164 чел.
  Месяц 27:       11,106 чел.
  Месяц 30:       12,136 чел.
  Месяц 33:       13,262 чел.
  Месяц 36:       14,491 чел.


### Интерпретация результата симуляции

При базовых параметрах **критическая масса НЕ достигается за 36 месяцев**:
- Через 36 месяцев в сети только **14 491 пользователь** — в 9 раз меньше критической массы.
- Причина: темп роста **3% в месяц** слишком медленный, чтобы за 3 года пройти путь от 5 000 до 133 333 пользователей.
- Математически при 3% роста нужно ≈ 111 месяцев (9+ лет), чтобы достичь критической массы.

**Это ключевой вывод:** стратегия компании по умолчанию **не работает в заданные сроки**.

In [5]:
# Сколько нужно месяцев, чтобы достичь критической массы при 3% роста?
import math

months_needed_3pct = math.log(critical_mass / N0) / math.log(1 + growth_before)
print(f"При темпе 3% в месяц критическая масса будет достигнута через:")
print(f"  {months_needed_3pct:.1f} месяцев ≈ {months_needed_3pct / 12:.1f} лет")
print()

# Какой темп роста нужен, чтобы достичь критической массы за 36 месяцев?
required_growth = (critical_mass / N0) ** (1 / horizon) - 1
print(f"Чтобы достичь критической массы за {horizon} месяцев, нужен темп роста:")
print(f"  ≈ {required_growth:.2%} в месяц")
print()
print(f"Это в {required_growth / growth_before:.1f} раза быстрее текущего темпа 3%")

При темпе 3% в месяц критическая масса будет достигнута через:
  111.1 месяцев ≈ 9.3 лет

Чтобы достичь критической массы за 36 месяцев, нужен темп роста:
  ≈ 9.55% в месяц

Это в 3.2 раза быстрее текущего темпа 3%


### Промежуточный вывод по критической массе

| Показатель | Значение |
|---|---|
| Критическая масса | 133 333 чел. |
| Через 36 месяцев при 3% роста | 14 491 чел. (11% от цели) |
| Месяцев до критической массы при 3% | ≈ 111 мес. (9.3 года) |
| Нужный темп роста для достижения за 36 мес. | ≈ 9.5% в месяц |

**Вывод:** для успешного запуска сети «ГородСоцСеть» необходимо **ускорить рост минимум в 3 раза** или изменить условия задачи (ARPU, CAC, fixed_costs).

## Часть 2. Оценка ценности сети

### Закон Мэткалфа

Ценность социальной сети **пропорциональна квадрату числа пользователей**:

(N) = k × N²
где:
- V(N) — ценность сети (руб.);
- N — число пользователей;
- k — коэффициент ценности (0.005 руб./пользователь²).

### Доход и прибыль

- **Доход от пользователей:** R(N) = ARPU × N
- **Прибыль сети:** П(N) = R(N) − fixed_costs − CAC × ΔN

где ΔN — новые пользователи за месяц (только за них платим CAC).

In [7]:
# Часть 2. Оценка ценности сети и расчёт прибыли по месяцам

# Инициализируем массивы для расчётов
values = []         # ценность по Мэткfалфу V(N) = k × N²
revenues = []       # доход R(N) = ARPU × N
cac_costs = []      # затраты на привлечение новых
profits = []        # прибыль П(N) = R - fixed_costs - CAC×ΔN

for i in range(len(users)):
    n = users[i]
    
    # Ценность сети по Мэткfалфу
    v = k * n ** 2
    values.append(v)
    
    # Доход от пользователей
    rev = ARPU * n
    revenues.append(rev)
    
    # Затраты на привлечение новых пользователей (только для i > 0)
    if i == 0:
        cac_cost = 0
    else:
        new_users = users[i] - users[i-1]
        cac_cost = CAC * new_users
    cac_costs.append(cac_cost)
    
    # Прибыль
    profit = rev - fixed_costs - cac_cost
    profits.append(profit)

# Вывод динамики по кварталам
print("Динамика показателей по кварталам:")
print(f"{'Месяц':>6} {'Пользов.':>12} {'Ценность, млн':>16} {'Доход, млн':>14} {'Прибыль, млн':>16}")
print("-" * 70)

for m in range(0, horizon + 1, 3):
    print(f"{m:>6} {users[m]:>12,.0f} {values[m]/1e6:>16,.2f} {revenues[m]/1e6:>14,.2f} {profits[m]/1e6:>16,.2f}")

Динамика показателей по кварталам:
 Месяц     Пользов.    Ценность, млн     Доход, млн     Прибыль, млн
----------------------------------------------------------------------
     0        5,000             0.12           0.07            -1.93
     3        5,464             0.15           0.08            -2.00
     6        5,970             0.18           0.09            -2.00
     9        6,524             0.21           0.10            -2.00
    12        7,129             0.25           0.11            -2.00
    15        7,790             0.30           0.12            -2.00
    18        8,512             0.36           0.13            -2.00
    21        9,301             0.43           0.14            -2.00
    24       10,164             0.52           0.15            -2.00
    27       11,106             0.62           0.17            -2.00
    30       12,136             0.74           0.18            -1.99
    33       13,262             0.88           0.20            -1.

### Интерпретация ценности сети

**Наблюдения за 36 месяцев:**

1. **Ценность сети по Мэткалфу растёт квадратично** — от 0.12 до 1.05 млн руб. — но в абсолюте пока очень мала.
2. **Доход покрывает лишь ~10% фиксированных затрат** к 36-му месяцу.
3. **Прибыль устойчиво отрицательная** — около −2 млн руб./мес. на протяжении всего горизонта.
4. **Причина:** сеть не достигла критической массы, поэтому операционные затраты не покрываются.

**Вывод:** без резкого ускорения роста проект экономически нежизнеспособен.

## Часть 3. Анализ эффективности: расчёт NPV

### Формула NPV
NPV = Σ [CF_t / (1 + r)^t]
где:
- CF_t — денежный поток в месяц t (у нас это **прибыль**);
- r = 0.01 — ставка дисконтирования (1% в месяц);
- горизонт = 36 месяцев.

### Что показывает NPV

- **NPV > 0** — проект эффективен.
- **NPV < 0** — проект убыточен.
- **NPV = 0** — проект на грани окупаемости.

In [8]:
# Часть 3. Расчёт NPV проекта за 36 месяцев

# Дисконтирование прибыли
npv = sum(profits[t] / (1 + discount_rate) ** t for t in range(len(profits)))

print(f"Расчёт NPV за {horizon} месяцев:")
print(f"  Ставка дисконтирования: {discount_rate:.0%} в месяц")
print(f"  Сумма прибыли без дисконтирования: {sum(profits):,.0f} руб.")
print(f"  NPV (дисконтированный):            {npv:,.0f} руб.")
print()

if npv > 0:
    print("✅ NPV > 0: проект эффективен.")
elif npv < 0:
    print("❌ NPV < 0: проект убыточен при базовых параметрах.")
else:
    print("⚠️ NPV = 0: проект на грани окупаемости.")


Расчёт NPV за 36 месяцев:
  Ставка дисконтирования: 1% в месяц
  Сумма прибыли без дисконтирования: -73,782,629 руб.
  NPV (дисконтированный):            -62,024,621 руб.

❌ NPV < 0: проект убыточен при базовых параметрах.


### Что это значит для бизнеса

**При базовых параметрах проект «ГородСоцСеть» экономически нежизнеспособен:**
- NPV отрицательный.
- Каждый месяц убыток около 2 млн руб.
- Компания тратит больше, чем зарабатывает, и не может догнать критическую массу.

**Это ключевой вывод для Части 4:** нужно либо ускорять рост, либо менять экономику (ARPU/CAC/fixed_costs), либо менять бизнес-модель.

## Анализ чувствительности

Проверим, как NPV меняется при разных сценариях:
- **Базовый:** 3% до крит. массы, 15% после.
- **Пессимистичный:** 2% до крит. массы, 10% после.
- **Оптимистичный:** 5% до крит. массы, 20% после.

Также проверим, при каком **ARPU** проект становится безубыточным.

In [9]:
# Функция: расчёт NPV при разных темпах роста и параметрах
def calculate_npv_nn(growth_before_test, growth_after_test, 
                     arpu_test=None, cac_test=None, fixed_costs_test=None):
    """Симулирует рост сети и возвращает NPV при заданных параметрах."""
    if arpu_test is None:
        arpu_test = ARPU
    if cac_test is None:
        cac_test = CAC
    if fixed_costs_test is None:
        fixed_costs_test = fixed_costs
    
    # Симуляция роста
    users_t = [N0]
    for m in range(1, horizon + 1):
        prev = users_t[-1]
        growth = growth_before_test if prev < critical_mass else growth_after_test
        new_n = min(prev * (1 + growth), M)
        users_t.append(new_n)
    
    # Расчёт прибыли по месяцам
    profits_t = []
    for i in range(len(users_t)):
        n = users_t[i]
        rev = arpu_test * n
        if i == 0:
            cac_cost = 0
        else:
            cac_cost = cac_test * (users_t[i] - users_t[i-1])
        profit = rev - fixed_costs_test - cac_cost
        profits_t.append(profit)
    
    # NPV
    npv_t = sum(profits_t[t] / (1 + discount_rate) ** t for t in range(len(profits_t)))
    return npv_t, users_t[-1]

# Три сценария
print("Анализ чувствительности NPV к темпу роста:")
print("-" * 75)
print(f"{'Сценарий':<25} {'Рост до':>10} {'Рост после':>12} {'Польз. за 36м':>15} {'NPV, млн':>14}")
print("-" * 75)

scenarios = [
    ("Пессимистичный", 0.02, 0.10),
    ("Базовый",        0.03, 0.15),
    ("Оптимистичный",  0.05, 0.20),
]

npv_results = {}
for name, g_before, g_after in scenarios:
    npv_sc, users_end = calculate_npv_nn(g_before, g_after)
    npv_results[name] = npv_sc
    print(f"{name:<25} {g_before:>9.0%} {g_after:>11.0%} {users_end:>15,.0f} {npv_sc/1e6:>13.1f}")

print("-" * 75)


Анализ чувствительности NPV к темпу роста:
---------------------------------------------------------------------------
Сценарий                     Рост до   Рост после   Польз. за 36м       NPV, млн
---------------------------------------------------------------------------
Пессимистичный                   2%         10%          10,199         -61.0
Базовый                          3%         15%          14,491         -62.0
Оптимистичный                    5%         20%          28,959         -65.7
---------------------------------------------------------------------------


### Парадокс анализа чувствительности

| Сценарий | Рост до | Рост после | NPV, млн руб. |
|---|---|---|---|
| Пессимистичный | 2% | 10% | **−61.0** |
| Базовый | 3% | 15% | **−62.0** |
| Оптимистичный | 5% | 20% | **−65.7** |

**Удивительный результат:** чем **быстрее** растёт сеть, тем **хуже** NPV.

### Почему так происходит

Каждый новый пользователь требует затрат CAC = 500 руб., а приносит только ARPU = 15 руб./мес. За 36 месяцев он принесёт максимум 15 × 36 = 540 руб. дохода, но с учётом дисконтирования — меньше. Итог: **каждый новый пользователь в убыток**.

- **При 3% роста:** новые пользователи = 0.03 × N
- **При 5% роста:** новые пользователи = 0.05 × N
- **CAC-затраты растут пропорционально темпу роста.**

Доход R = ARPU × N растёт **медленнее**, чем CAC × ΔN при ускорении роста. Отсюда — ухудшение NPV.

### Экономический смысл

Это **классическая ловушка «покупки убыточного роста»**:
- Компания быстро набирает пользователей, но каждый из них стоит дороже, чем приносит.
- Ускорение роста **усиливает проблему**, а не решает её.
- **Решение:** снижать CAC, повышать ARPU или менять бизнес-модель.

In [10]:
# Поиск минимального ARPU, при котором NPV = 0
from scipy.optimize import brentq

def npv_for_arpu(arpu_test):
    npv_sc, _ = calculate_npv_nn(growth_before, growth_after, arpu_test=arpu_test)
    return npv_sc

# Проверяем границы: от 15 руб (убыток) до 500 руб (прибыль)
arpu_min = 15
arpu_max = 500

npv_at_min = npv_for_arpu(arpu_min)
npv_at_max = npv_for_arpu(arpu_max)

print(f"Проверка границ:")
print(f"  NPV при ARPU = {arpu_min} руб.: {npv_at_min/1e6:>10,.2f} млн руб.")
print(f"  NPV при ARPU = {arpu_max} руб.: {npv_at_max/1e6:>10,.2f} млн руб.")
print()

if npv_at_min * npv_at_max < 0:
    arpu_breakeven = brentq(npv_for_arpu, arpu_min, arpu_max)
    print(f"✅ Безубыточный ARPU: ≈ {arpu_breakeven:.2f} руб./пользователь/мес.")
    print(f"   Это в {arpu_breakeven/ARPU:.1f} раз выше текущего ARPU ({ARPU} руб.)")
else:
    print("⚠️ В диапазоне [15; 500] руб. NPV не меняет знак.")
    print("   Даже при высоком ARPU проект остаётся убыточным.")
    print("   Причина: фиксированные затраты 2 млн руб./мес. превышают")
    print("   все возможные доходы в рамках 36 месяцев.")

Проверка границ:
  NPV при ARPU = 15 руб.:     -62.02 млн руб.
  NPV при ARPU = 500 руб.:      68.49 млн руб.

✅ Безубыточный ARPU: ≈ 245.48 руб./пользователь/мес.
   Это в 16.4 раз выше текущего ARPU (15 руб.)


## Часть 4. Стратегические выводы

### 4.1. Как достичь критической массы на 6 месяцев раньше?

Сейчас критическая масса не достигается за 36 месяцев. Чтобы **сократить время до критической массы на 6 месяцев** относительно необходимого темпа 9.55%, нужно:

**Вариант А — ускорить темп роста:**
- Достичь роста выше 9.55%/мес. на ранних этапах.
- Возможно, за счёт агрессивного маркетинга (но CAC растёт — парадокс убыточного роста).

**Вариант Б — изменить экономику:**
- Увеличить ARPU (монетизация).
- Снизить CAC (виральный рост, реферальные программы).
- Снизить фиксированные затраты (оптимизация инфраструктуры).

In [11]:
# Часть 4.1. Сколько нужно инвестировать, чтобы ускориться на 6 месяцев?

# Сейчас критическая масса не достигается за 36 месяцев вообще.
# Значит, "ускорить на 6 месяцев" не имеет смысла в текущих параметрах.
# Но можно посчитать: сколько нужно вложить в маркетинг, чтобы достичь крит. массы за 30 месяцев.

target_months = 30

# Какой темп роста нужен для 30 месяцев?
required_growth_30 = (critical_mass / N0) ** (1 / target_months) - 1

print(f"Целевой срок достижения крит. массы: {target_months} мес.")
print(f"Требуемый темп роста: {required_growth_30:.2%} в месяц")
print()

# Симулируем рост с этим темпом, считаем дополнительных пользователей и CAC
users_fast = [N0]
for m in range(1, target_months + 1):
    prev = users_fast[-1]
    growth = required_growth_30 if prev < critical_mass else growth_after
    new_n = min(prev * (1 + growth), M)
    users_fast.append(new_n)

# Сравниваем: сколько пользователей "лишних" относительно базового
base_users_30 = users[30] if len(users) > 30 else users[-1]
fast_users_30 = users_fast[-1]
extra_users = fast_users_30 - base_users_30

# Инвестиции в маркетинг: CAC × суммарные новые пользователи
extra_cac_cost = CAC * extra_users

print(f"Пользователей в базовом сценарии к мес. {target_months}: {base_users_30:,.0f}")
print(f"Пользователей в ускоренном сценарии к мес. {target_months}: {fast_users_30:,.0f}")
print(f"Дополнительных пользователей: {extra_users:,.0f}")
print()
print(f"Дополнительные затраты на маркетинг (CAC × ΔN):")
print(f"  {CAC} × {extra_users:,.0f} = {extra_cac_cost:,.0f} руб.")
print()
print("⚠️ ВАЖНО: эта цифра в разы превышает возможную выручку за тот же период.")
print("Значит, при текущих ARPU и CAC простое увеличение маркетинга НЕ РАБОТАЕТ.")

Целевой срок достижения крит. массы: 30 мес.
Требуемый темп роста: 11.57% в месяц

Пользователей в базовом сценарии к мес. 30: 12,136
Пользователей в ускоренном сценарии к мес. 30: 133,333
Дополнительных пользователей: 121,197

Дополнительные затраты на маркетинг (CAC × ΔN):
  500 × 121,197 = 60,598,510 руб.

⚠️ ВАЖНО: эта цифра в разы превышает возможную выручку за тот же период.
Значит, при текущих ARPU и CAC простое увеличение маркетинга НЕ РАБОТАЕТ.


### 4.2. Стратегические рекомендации для «ГородСоцСеть»

На основе проведённого анализа можно выделить **пять ключевых направлений**:

#### 1. Сменить бизнес-модель монетизации

Текущий ARPU = 15 руб./мес. — **недостаточен** для окупаемости. Чтобы выйти на безубыточность, нужно ≈245 руб./пользователь/мес.

**Возможные варианты:**
- Ввести **премиум-подписку** (расширенные функции, без рекламы, аналитика для бизнеса).
- Использовать **рекламную модель** (таргетированная реклама, партнёрские программы).
- Развивать **B2B-направление** (сервисы для местного бизнеса, городских служб).
- Продавать **данные в обезличенном виде** (городская аналитика).

#### 2. Снизить стоимость привлечения (CAC)

Сейчас CAC = 500 руб. — это высокая стоимость для соцсети. **Реальные пути снижения:**
- **Виральные механики** (реферальные программы, приглашения друзей, семейные аккаунты).
- **Партнёрства** с местными компаниями (банки, ритейл, мобильные операторы).
- **Контент-маркетинг** и работа с сообществом вместо платной рекламы.

#### 3. Оптимизировать фиксированные затраты

2 млн руб./мес. — это 72 млн руб. за 3 года при текущей выручке всего 8–10 млн руб. Возможные шаги:
- Использовать **облачную инфраструктуру** с pay-as-you-go моделью.
- Начинать с **минимальной команды** (3–5 человек) и масштабироваться по мере роста.
- Перевести часть функций на **аутсорс**.

#### 4. Сфокусироваться на узком сегменте

Вместо запуска на весь миллионный город — **сначала захватить один район или сообщество** (студенты, ЖК, бизнес-ассоциация). Это:
- Снизит расходы на маркетинг.
- Ускорит достижение плотности связей.
- Позволит протестировать монетизацию до масштабирования.

#### 5. Реалистичные KPI

Вместо «3% роста и 36 месяцев до критической массы» поставить измеримые цели:
- **Первый квартал:** 10 000 пользователей, ARPU ≥ 30 руб.
- **Первый год:** 50 000 пользователей, CAC ≤ 200 руб.
- **Третий год:** выход на 200 000+ пользователей и положительный NPV.

### 4.3. Вывод

Проект «ГородСоцСеть» в **текущих параметрах убыточен** (NPV = −62 млн руб.). Причины:
1. **Низкий ARPU** (15 руб. против необходимых 245 руб.).
2. **Высокий CAC** (500 руб.).
3. **Большие фиксированные затраты** (2 млн руб./мес.).
4. **Медленный рост** — критическая масса недостижима за 36 месяцев.

**Ускорение роста без изменения экономики только усиливает убытки** («покупка убыточного роста»).

**Успех возможен только при комплексной смене стратегии:** монетизация, виральный рост, оптимизация затрат и фокус на узком сегменте.